# 01 — Tổng quan PTB Diagnostic ECG Database

PTB Diagnostic ECG Database là bộ dữ liệu điện tim nghỉ của Physikalisch-Technische Bundesanstalt. Trong bản dữ liệu đang dùng, mỗi recording có 12 chuyển đạo ECG chuẩn và 3 chuyển đạo Frank vectorcardiogram (`vx`, `vy`, `vz`).

Notebook này **chỉ khám phá dữ liệu**, không huấn luyện Machine Learning. Ta sẽ kiểm tra file thật, header, tín hiệu, metadata, bệnh nhân, diagnosis, độ dài, tần số lấy mẫu và mức độ đầy đủ của X/Y/Z. Mọi con số được tính khi chạy notebook; không coi mô tả trong README là kết quả thay cho dữ liệu đang có.


## 1. Thư viện

Nếu môi trường chưa có thư viện, bỏ dấu `#` ở dòng `%pip` và chạy cell một lần. Sau khi cài, nên restart kernel. `wfdb` là thư viện đọc định dạng WaveForm DataBase (`.hea`, `.dat`, `.xyz`).


In [ ]:
%pip install numpy pandas matplotlib wfdb

from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (12, 4)


## 2. Cấu hình đường dẫn

Hãy sửa `DATA_ROOT` nếu bạn đặt PTB ở nơi khác. Thư mục đúng là thư mục chứa các folder `patient001`, `patient002`, ... và file `RECORDS`. Đường dẫn dưới đây khớp workspace hiện tại, nhưng vẫn là một biến cấu hình, không bị giấu trong các function.


In [ ]:
DATA_ROOT = Path("content/ptbdb/physionet.org/files/ptbdb/1.0.0")

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy DATA_ROOT={DATA_ROOT.resolve()}. "
        "Hãy sửa DATA_ROOT tới thư mục PTB chứa các folder patient..."
    )
print("DATA_ROOT:", DATA_ROOT.resolve())


## 3. Các function đọc dữ liệu an toàn

- Record được nhận diện từ file `.hea`; không đoán trước tên record.
- Metadata là các dòng bắt đầu bằng `#` trong header.
- Tên field được giữ nguyên. Nếu một field lặp lại trong cùng header, thêm hậu tố `__2`, `__3` để không silently ghi đè.
- `patient_id` ưu tiên lấy từ tên folder `patient...`; nếu cấu trúc khác, dùng tên folder cha và phát cảnh báo ở bảng kết quả.


In [ ]:
def find_record_bases(data_root):
    """Input: DATA_ROOT. Output: danh sách record base suy ra từ toàn bộ file .hea."""
    return sorted(path.with_suffix("") for path in Path(data_root).rglob("*.hea"))


def read_header_text(record_base):
    """Input: record base. Output: nội dung raw của file .hea dưới dạng chuỗi."""
    return Path(record_base).with_suffix(".hea").read_text(encoding="latin-1")


def parse_header_metadata(header_text):
    """Input: raw header. Output: dictionary field-value, bảo toàn các field lặp."""
    parsed = {}
    counts = Counter()
    for raw_line in header_text.splitlines():
        line = raw_line.strip()
        if not line.startswith("#"):
            continue
        content = line[1:].strip()
        if ":" not in content:
            continue
        field, value = (part.strip() for part in content.split(":", 1))
        if not field:
            continue
        counts[field] += 1
        key = field if counts[field] == 1 else f"{field}__{counts[field]}"
        parsed[key] = value or np.nan
    return parsed


def patient_id_from_path(record_base):
    """Input: record path. Output: patient_id lấy từ thành phần patientNNN hoặc folder cha."""
    for part in reversed(Path(record_base).parts):
        if re.fullmatch(r"patient\d+", part, flags=re.IGNORECASE):
            return part.lower()
    return Path(record_base).parent.name


def find_xyz_indices(channel_names):
    """Input: danh sách channel. Output: index duy nhất của Frank X, Y, Z hoặc None."""
    aliases = {"X": {"x", "vx", "frankx", "frank_x"},
               "Y": {"y", "vy", "franky", "frank_y"},
               "Z": {"z", "vz", "frankz", "frank_z"}}
    normalized = [re.sub(r"[^a-z0-9_]", "", str(name).lower()) for name in channel_names]
    found = {}
    for axis, accepted in aliases.items():
        hits = [i for i, name in enumerate(normalized) if name in accepted]
        found[axis] = hits[0] if len(hits) == 1 else None
    return found


record_bases = find_record_bases(DATA_ROOT)
if not record_bases:
    raise RuntimeError("Không tìm thấy file .hea nào trong DATA_ROOT.")
print(f"Tìm thấy {len(record_bases):,} record base từ file .hea")


## 4. Quét toàn bộ dataset

`.hea` chứa cấu trúc và metadata; `.dat` chứa 12 ECG; `.xyz` chứa Frank X/Y/Z trong bản PTB này. Một record được đếm theo một `.hea`, không phải theo tổng số file.


In [ ]:
all_files = [p for p in DATA_ROOT.rglob("*") if p.is_file()]
files_by_suffix = pd.Series([p.suffix.lower() or "(không đuôi)" for p in all_files]).value_counts()

print(f"Tổng số file: {len(all_files):,}")
display(files_by_suffix.rename("n_files").to_frame())
print(f"Tổng số record (.hea): {len(record_bases):,}")
print("Một vài record đầu:")
for base in record_bases[:8]:
    print(" -", base.relative_to(DATA_ROOT))


## 5. Đọc một record mẫu

`sig_name` là tên các kênh; `fs` là số mẫu mỗi giây (Hz); `p_signal.shape = (số thời điểm, số kênh)`. Vì vậy mỗi hàng là một thời điểm và mỗi cột là một channel.


In [ ]:
SAMPLE_RECORD = record_bases[0]  # lựa chọn xác định, không random
sample = wfdb.rdrecord(str(SAMPLE_RECORD))

print("Record:", SAMPLE_RECORD.relative_to(DATA_ROOT))
print("sig_name:", sample.sig_name)
print("fs:", sample.fs, "Hz")
print("p_signal.shape:", sample.p_signal.shape)
print("units:", sample.units)


## 6. Liệt kê channel và phát hiện Frank X/Y/Z


In [ ]:
xyz_idx = find_xyz_indices(sample.sig_name)
channel_table = pd.DataFrame({
    "index": range(len(sample.sig_name)),
    "channel": sample.sig_name,
    "unit": sample.units,
})
channel_table["Frank_axis"] = ""
for axis, idx in xyz_idx.items():
    if idx is not None:
        channel_table.loc[idx, "Frank_axis"] = axis
display(channel_table)
print("Các index tự phát hiện:", xyz_idx)


## 7. Vẽ 5 giây tín hiệu gốc

Ta chọn lead II nếu có, nếu không dùng ECG channel đầu tiên; X/Y/Z luôn được tìm bằng tên. Mỗi đồ thị riêng để dễ nhìn biên độ và đơn vị.


In [ ]:
seconds_to_plot = 5
n_plot = min(len(sample.p_signal), int(seconds_to_plot * sample.fs))
t = np.arange(n_plot) / sample.fs
lower_names = [name.lower() for name in sample.sig_name]
ecg_idx = lower_names.index("ii") if "ii" in lower_names else 0
plot_items = [(sample.sig_name[ecg_idx], ecg_idx)] + [
    (f"Frank {axis} ({sample.sig_name[idx]})", idx)
    for axis, idx in xyz_idx.items() if idx is not None
]

for title, idx in plot_items:
    plt.figure(figsize=(12, 3))
    plt.plot(t, sample.p_signal[:n_plot, idx], linewidth=0.8)
    plt.title(f"{title} — {n_plot / sample.fs:.1f} giây đầu")
    plt.xlabel("Thời gian (giây)")
    plt.ylabel(sample.units[idx])
    plt.grid(alpha=0.25)
    plt.show()


## 8. Raw header và metadata đã parse

Hiển thị raw trước giúp kiểm chứng parser. Không giả định sẵn tuổi, thuốc hay vị trí MI luôn tồn tại. Parser chỉ lấy những dòng thực sự xuất hiện.


In [ ]:
sample_header_text = read_header_text(SAMPLE_RECORD)
print(sample_header_text)

sample_metadata = parse_header_metadata(sample_header_text)
display(pd.DataFrame(sample_metadata.items(), columns=["field", "value"]))


## 9. Thống kê toàn bộ metadata field

Các tên khác nhau vẫn nằm ở các dòng khác nhau; notebook **không tự merge**. `tỷ lệ có dữ liệu` tính trên tổng số record, với chuỗi rỗng/`n/a`/`-` được coi là thiếu cho thống kê độ đầy đủ (raw value vẫn được giữ trong bảng master).


In [ ]:
metadata_per_record = []
for base in record_bases:
    metadata_per_record.append(parse_header_metadata(read_header_text(base)))

missing_tokens = {"", "n/a", "na", "nan", "none", "-", "unknown"}
all_fields = sorted({field for meta in metadata_per_record for field in meta})
rows = []
for field in all_fields:
    raw_values = [meta.get(field, np.nan) for meta in metadata_per_record]
    present = [v for v in raw_values if pd.notna(v)]
    usable = [v for v in present if str(v).strip().lower() not in missing_tokens]
    rows.append({
        "field": field,
        "số record có field": len(present),
        "tỷ lệ có dữ liệu (%)": 100 * len(usable) / len(record_bases),
        "ví dụ giá trị": next(iter(usable), np.nan),
    })

df_metadata_fields = pd.DataFrame(rows).sort_values(
    ["số record có field", "field"], ascending=[False, True]
).reset_index(drop=True)
print(f"Số tên field metadata phân biệt: {len(df_metadata_fields)}")
display(df_metadata_fields)


## 10. Tạo bảng master metadata: `df_records`

Mỗi hàng là một recording. `diagnosis` lấy đúng từ field `Reason for admission`; nếu bản dữ liệu khác không có field này, giá trị là thiếu thay vì suy đoán từ field khác.


In [ ]:
record_rows = []
read_errors = []
for base, meta in zip(record_bases, metadata_per_record):
    try:
        header = wfdb.rdheader(str(base))
        relative_id = base.relative_to(DATA_ROOT).as_posix()
        row = {
            "record_id": relative_id,
            "patient_id": patient_id_from_path(base),
            "diagnosis": meta.get("Reason for admission", np.nan),
            "fs": float(header.fs),
            "n_samples": int(header.sig_len),
            "duration_seconds": float(header.sig_len / header.fs),
            "n_channels": int(header.n_sig),
            "channel_names": list(header.sig_name),
        }
        row.update(meta)
        record_rows.append(row)
    except Exception as exc:
        read_errors.append({"record_id": str(base), "error": repr(exc)})

df_records = pd.DataFrame(record_rows)
print("df_records.shape:", df_records.shape)
display(df_records.head())
if read_errors:
    print(f"CẢNH BÁO: {len(read_errors)} header lỗi")
    display(pd.DataFrame(read_errors))


## 11. Thống kê bệnh nhân và số recording mỗi bệnh nhân


In [ ]:
patient_counts = df_records.groupby("patient_id").size().rename("n_records")
df_patients = patient_counts.reset_index()
patient_diagnoses = df_records.groupby("patient_id")["diagnosis"].agg(
    lambda values: sorted({str(v) for v in values if pd.notna(v)})
)
df_patients = df_patients.merge(patient_diagnoses.rename("diagnoses"), on="patient_id")

print("Tổng số bệnh nhân:", df_records["patient_id"].nunique())
print("Tổng số recording:", len(df_records))
display(patient_counts.describe()[["min", "max", "mean", "50%"]].rename({"50%": "median"}).to_frame("records/patient"))

plt.figure(figsize=(9, 4))
bins = np.arange(patient_counts.min(), patient_counts.max() + 2) - 0.5
plt.hist(patient_counts, bins=bins, edgecolor="black")
plt.xlabel("Số recording / bệnh nhân")
plt.ylabel("Số bệnh nhân")
plt.title("Phân bố số recording theo bệnh nhân")
plt.xticks(range(patient_counts.min(), patient_counts.max() + 1))
plt.show()


## 12. Diagnosis thực tế


In [ ]:
diagnosis_table = (
    df_records.groupby("diagnosis", dropna=False)
    .agg(n_records=("record_id", "size"), n_patients=("patient_id", "nunique"))
    .sort_values("n_records", ascending=False)
    .reset_index()
)
display(diagnosis_table)

plot_diagnosis = diagnosis_table.fillna({"diagnosis": "(missing)"})
plt.figure(figsize=(11, max(4, 0.35 * len(plot_diagnosis))))
plt.barh(plot_diagnosis["diagnosis"].astype(str), plot_diagnosis["n_records"])
plt.gca().invert_yaxis()
plt.xlabel("Số recording")
plt.title("Số recording theo diagnosis")
plt.show()


## 13. Mapping MI / HC / OTHER

Trước khi mapping, bảng trên đã in mọi diagnosis thực tế. Mapping dưới đây chỉ dùng **so khớp chính xác sau khi chuẩn hóa chữ thường và khoảng trắng**. Không dùng tìm kiếm mơ hồ như `"infarction" in diagnosis`.


In [ ]:
MI_DIAGNOSES = {"myocardial infarction"}
HC_DIAGNOSES = {"healthy control", "healthy controls"}

def map_diagnosis_group(value):
    """Input: diagnosis raw. Output: MI, HC hoặc OTHER bằng exact mapping."""
    if pd.isna(value):
        return "OTHER"
    normalized = " ".join(str(value).strip().lower().split())
    if normalized in MI_DIAGNOSES:
        return "MI"
    if normalized in HC_DIAGNOSES:
        return "HC"
    return "OTHER"

df_records["group"] = df_records["diagnosis"].map(map_diagnosis_group)
group_summary = (
    df_records.groupby("group")
    .agg(n_patients=("patient_id", "nunique"), n_records=("record_id", "size"))
    .reindex(["MI", "HC", "OTHER"], fill_value=0)
)
display(group_summary)


## 14. Mất cân bằng: record-level và patient-level

Record-level đếm mỗi lần đo; patient-level đếm mỗi người đúng một lần trong từng nhóm. Hai số khác nhau vì một bệnh nhân có thể có nhiều recording. Đây cũng là lý do notebook 2 phải split theo `patient_id`.


In [ ]:
mi_hc = df_records[df_records["group"].isin(["MI", "HC"])]
record_balance = mi_hc["group"].value_counts().reindex(["MI", "HC"], fill_value=0)
patient_balance = mi_hc.groupby("group")["patient_id"].nunique().reindex(["MI", "HC"], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
record_balance.plot.bar(ax=axes[0], title="MI vs HC — theo recording", color=["#c44e52", "#4c72b0"])
patient_balance.plot.bar(ax=axes[1], title="MI vs HC — theo bệnh nhân", color=["#c44e52", "#4c72b0"])
for ax in axes:
    ax.set_ylabel("Số lượng")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 15. Độ dài recording


In [ ]:
display(df_records["duration_seconds"].describe()[["min", "max", "mean", "50%"]].rename({"50%": "median"}).to_frame())
plt.figure(figsize=(9, 4))
plt.hist(df_records["duration_seconds"], bins=30, edgecolor="black")
plt.xlabel("Độ dài (giây)")
plt.ylabel("Số recording")
plt.title("Phân bố độ dài recording")
plt.show()


## 16. Tần số lấy mẫu — kiểm tra từ dữ liệu, không hard-code


In [ ]:
fs_table = df_records["fs"].value_counts().sort_index().rename_axis("fs_Hz").to_frame("n_records")
display(fs_table)
print("Tất cả record có 1000 Hz?", bool((df_records["fs"] == 1000).all()))


## 17. Kiểm tra Frank X/Y/Z trên mọi record


In [ ]:
xyz_rows = []
for base in record_bases:
    try:
        header = wfdb.rdheader(str(base))
        found = find_xyz_indices(header.sig_name)
        xyz_rows.append({
            "record": base.relative_to(DATA_ROOT).as_posix(),
            "X exists": found["X"] is not None,
            "Y exists": found["Y"] is not None,
            "Z exists": found["Z"] is not None,
            "channel names": list(header.sig_name),
        })
    except Exception as exc:
        xyz_rows.append({"record": str(base), "X exists": False, "Y exists": False,
                         "Z exists": False, "channel names": f"ERROR: {exc}"})

df_xyz_check = pd.DataFrame(xyz_rows)
display(df_xyz_check.head())
print("Số record thiếu X/Y/Z:", (~df_xyz_check[["X exists", "Y exists", "Z exists"]].all(axis=1)).sum())
display(df_xyz_check.loc[~df_xyz_check[["X exists", "Y exists", "Z exists"]].all(axis=1)])


## 18. Biến cần cho dự án MI vs HC


In [ ]:
metadata_presence = {field: field in df_records.columns for field in ["age", "sex", "Medication pre admission", "Acute infarction (localization)"]}
variable_plan = pd.DataFrame([
    ["patient_id", True, True, "Nhóm các recording và split train/test; không đưa vào ML"],
    ["record_id", True, True, "Truy vết recording; không đưa vào ML"],
    ["diagnosis", True, True, "Tạo label MI/HC; không đưa vào ML"],
    ["fs", True, True, "Đổi mili-giây sang số sample và tính vận tốc"],
    ["X", bool(df_xyz_check["X exists"].all()), True, "Frank VCG, input tín hiệu chính"],
    ["Y", bool(df_xyz_check["Y exists"].all()), True, "Frank VCG, input tín hiệu chính"],
    ["Z", bool(df_xyz_check["Z exists"].all()), True, "Frank VCG, input tín hiệu chính"],
    ["age", metadata_presence["age"], False, "Mô tả; không dùng làm input ML cơ bản"],
    ["sex", metadata_presence["sex"], False, "Mô tả; không dùng làm input ML cơ bản"],
    ["Medication pre admission", metadata_presence["Medication pre admission"], False, "Metadata lâm sàng; không dùng làm input ML cơ bản"],
    ["Acute infarction (localization)", metadata_presence["Acute infarction (localization)"], False, "Mô tả MI; không dùng làm input ML cơ bản"],
], columns=["Variable", "Có sẵn trong PTB?", "Dùng cho pipeline?", "Vai trò"])
display(variable_plan)


## 19. Tóm tắt và export


In [ ]:
n_diagnoses = df_records["diagnosis"].nunique(dropna=True)
print(f"Dataset có {df_records['patient_id'].nunique():,} bệnh nhân")
print(f"Dataset có {len(df_records):,} recording")
print(f"Có {n_diagnoses:,} diagnosis không thiếu")
for group in ["MI", "HC", "OTHER"]:
    row = group_summary.loc[group]
    print(f"{group}: {row['n_patients']:,} bệnh nhân, {row['n_records']:,} recording")
print("VCG dùng cho dự án: X/Y/Z (phát hiện theo tên channel)")
print("Các biến gốc bắt buộc: patient_id, record_id, diagnosis, fs, X, Y, Z")

df_records.to_csv("ptb_records_overview.csv", index=False)
df_patients.to_csv("ptb_patients_overview.csv", index=False)
print("Đã lưu ptb_records_overview.csv và ptb_patients_overview.csv")
